# FIAP — MBA em Data Engineering
## Disciplina: Agents and Agentic AI
### Professor: Felipe Gustavo Silva Teodoro
### Integrantes do Grupo:
- **Fátima Beatriz Rodrigues**
- **Jean Felipe Ertzogue**
- **Luiz Soldatelli Neto**
- **Ricardo Peixoto**

---

# 🚀 Trabalho Final: QuantumFinance AI Investment Assistant
### *Assistente de Investimentos Autônomo e Explicável Baseado em AI Agents*

---

## 1. Contexto de Negócio & O Desafio

A **QuantumFinance** está expandindo sua atuação no mercado de capitais e renda variável. Para se destacar e entregar valor real aos clientes, a instituição precisa de um **Assistente de Investimentos inteligente** capaz de analisar o mercado e emitir recomendações autônomas de compra, venda ou manutenção de ações da B3 de forma **explicável, auditável e fundamentada**.

### Ações Prioritárias Monitoradas:
- 🔵 **VALE3** — Vale S.A. (Setor de Mineração)
- 🟢 **PETR4** — Petrobras (Setor de Energia / Óleo & Gás)
- 🟡 **BBAS3** — Banco do Brasil (Setor Financeiro)
- 🟠 **ITUB4** — Itaú Unibanco (Setor Financeiro)

---

## 2. Arquitetura da Solução (Padrão ReAct & Google ADK)

O assistente foi desenhado seguindo rigorosamente o paradigma **ReAct (Reasoning + Acting)** ensinado pelo Professor Felipe Teodoro, utilizando o **Google ADK** (`google-adk`). O agente não segue um pipeline engessado; ele raciocina passo a passo (*Chain-of-Thought*), aciona ferramentas de percepção (*Tools*), interpreta observações e sintetiza recomendações acionáveis.

As ferramentas seguem a nomenclatura sugerida no enunciado (slide 6): `search_news(ticker)`, `get_price_data(ticker, period)`, `calculate_indicators(...)` e `generate_recommendation(analysis)`, complementadas por `run_backtest_strategy` e `plot_technical_chart`.

```mermaid
flowchart TD
    User([Investidor / Usuário Final]) --> UI["Interface Conversacional\nGradio Chat / Jupyter"]

    subgraph Core [Núcleo do AI Agent - Google ADK]
        Runner["Runner & Session Manager\nInMemorySessionService"]
        Agent["Orquestrador ReAct Agent\nGoogle ADK + LiteLLM / Gemini"]
        Memory[("Histórico de Sessão\nMulti-Turn Memory")]
    end

    UI <-->|Pergunta / Resposta| Runner
    Runner <--> Agent
    Agent <--> Memory

    subgraph Perception [Percepção]
        T1["get_price_data(ticker, period)\nOHLCV via yfinance"]
        T2["calculate_indicators(ticker, period)\nRSI, MACD, SMA/EMA, Bollinger, Volume"]
        T3["search_news(ticker)\nRSS + Sentimento via LLM + Resumo"]
    end

    subgraph Action [Ação]
        T4["generate_recommendation(analysis)\nCOMPRAR / VENDER / AGUARDAR"]
        T5["run_backtest_strategy(ticker, dias)\nAcurácia + Backtest vs Buy & Hold"]
        T6["plot_technical_chart(ticker, period)\nGráfico Plotly"]
    end

    Agent -->|Thought + Action| Perception
    Perception -->|Observation| Agent
    Agent -->|Thought + Action| Action
    Action -->|Observation| Agent

    subgraph ExternalSources [Fontes Externas de Dados]
        YF[("Yahoo Finance\nCotações B3 .SA")]
        RSS[("Feeds RSS\nGoogle News, InfoMoney")]
        LLM[("LLM\nGemini / GPT")]
    end

    T1 <--> YF
    T2 <--> YF
    T3 <--> RSS
    T3 <--> LLM
    T5 <--> YF
    T6 <--> YF
```


## 3. Preparação do Ambiente & Instalação de Dependências

Instalação do **Google ADK**, **LiteLLM**, **yfinance**, **feedparser**, bibliotecas estatísticas e o **Gradio** para a interface interativa.


In [ ]:
# Instalação das dependências necessárias para execução no Google Colab, Databricks ou ambiente local
%pip install -q \
    google-adk \
    google-genai \
    litellm \
    yfinance \
    feedparser \
    pandas \
    numpy \
    plotly \
    gradio \
    requests


In [ ]:
import os
import getpass
import warnings
import json
import re
import time
import inspect
from datetime import datetime, timedelta
import requests
import feedparser
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import gradio as gr
from IPython.display import Markdown, display

# Desativa avisos de depreciação de bibliotecas auxiliares
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print("✅ Ambiente configurado com sucesso!")


## 4. Configuração Segura de Credenciais & Gestão de Segredos (Databricks + Colab)

Seguindo as melhores práticas de governança de dados, segurança da informação e arquitetura de missão crítica:
- O agente aceita chaves de API (`GEMINI_API_KEY` ou `OPENAI_API_KEY`) consultando em ordem hierárquica:
  1. **Databricks Secret Scope** (`dbutils.secrets`): para clusters corporativos ou Community Edition no Databricks.
  2. **Google Colab Secrets** (`google.colab.userdata`): para execuções no ecossistema Colab.
  3. **Variáveis de ambiente nativas** (`os.environ`).
  4. **Entrada mascarada e interativa** (`getpass`).

### 🛡️ Diferencial Arquitetural: Graceful Degradation (Zero-Token Dependency)
Em sistemas corporativos de Engenharia de Dados, **um agente não pode travar ou interromper pipelines** caso a API da LLM atinja *rate limits* (HTTP 429), fique indisponível ou caso o usuário não possua saldo no momento.
- Se houver chave: o agente executa a inferência neural via **Google ADK** (`gemini-2.5-flash` ou `gpt-4o-mini`) e a classificação de sentimento das notícias também é feita pela LLM.
- Se não houver chave (ou em caso de timeout): o agente ativa automaticamente o **Modo Resiliente (Motor Analítico Determinístico)**, consumindo os dados reais da B3 (`yfinance`), calculando os indicadores, processando os feeds RSS (`feedparser`) com um léxico financeiro e emitindo a recomendação com Chain-of-Thought e o JSON canônico do Slide 7.
- **Benefício**: Garante **100% de reprodutibilidade** no **Google Colab**, **Databricks Community Edition (Free)** ou ambiente local para qualquer avaliador!


In [ ]:
# Configuração segura e universal de chaves de API (Google Gemini ou OpenAI)
# Suporta Databricks Secret Scope, Colab Secrets, os.environ e getpass mascarado.

MODEL_PROVIDER = "gemini" # Opções: "gemini" ou "openai"
SECRET_SCOPE_DATABRICKS = "quantumfinance"


def obter_segredo(nome_chave: str, nome_secret_databricks: str) -> str:
    """Busca o segredo no Databricks, Colab, os.environ ou getpass de forma resiliente."""
    # 1. Tentar Databricks Secret Scope (dbutils.secrets)
    try:
        # No Databricks, dbutils está no namespace global da sessão
        if "dbutils" in globals() or "dbutils" in locals():
            d_utils = globals().get("dbutils") or locals().get("dbutils")
            segredo = d_utils.secrets.get(scope=SECRET_SCOPE_DATABRICKS, key=nome_secret_databricks)
            if segredo:
                print(f"🔑 Segredo '{nome_secret_databricks}' carregado via Databricks Secret Scope ('{SECRET_SCOPE_DATABRICKS}')!")
                return segredo
    except Exception:
        pass

    # 2. Tentar Google Colab Secrets (google.colab.userdata)
    try:
        from google.colab import userdata
        segredo = userdata.get(nome_chave)
        if segredo:
            print(f"🔑 Segredo '{nome_chave}' carregado via Google Colab Secrets!")
            return segredo
    except Exception:
        pass

    # 3. Tentar variáveis de ambiente nativas (os.environ)
    if os.environ.get(nome_chave):
        print(f"🔑 Segredo '{nome_chave}' encontrado em os.environ!")
        return os.environ[nome_chave]
        
    # Alias específico do Gemini/Google
    if nome_chave == "GEMINI_API_KEY" and os.environ.get("GOOGLE_API_KEY"):
        print("🔑 Segredo 'GOOGLE_API_KEY' encontrado em os.environ!")
        return os.environ["GOOGLE_API_KEY"]

    # 4. Fallback interativo mascarado (getpass)
    print(f"Informe a sua {nome_chave} (ou pressione Enter para modo demonstração):")
    try:
        return getpass.getpass()
    except Exception:
        return ""


if MODEL_PROVIDER == "gemini":
    chave = obter_segredo("GEMINI_API_KEY", "gemini_api_key")
    if chave:
        os.environ["GEMINI_API_KEY"] = chave
        os.environ["GOOGLE_API_KEY"] = chave
        print("✅ GEMINI_API_KEY registrada com sucesso em os.environ!")
    else:
        print("⚠️ Nenhuma chave informada. O agente funcionará no modo de demonstração analítica.")
else:
    chave = obter_segredo("OPENAI_API_KEY", "openai_api_key")
    if chave:
        os.environ["OPENAI_API_KEY"] = chave
        print("✅ OPENAI_API_KEY registrada com sucesso em os.environ!")
    else:
        print("⚠️ Nenhuma chave informada. O agente funcionará no modo de demonstração analítica.")


## 5. Módulo de Percepção: Ferramentas de Coleta e Análise

O agente dispõe de seis ferramentas com tipagem estrita e docstrings (usadas pelo Google ADK como descrição para a LLM). Os nomes seguem o fluxo sugerido no slide 6 do enunciado:

| # | Ferramenta | Dimensão | Função |
|:-:|:--|:--|:--|
| 1 | `get_price_data(ticker, period)` | Percepção | Preços e volumes OHLCV da B3 via `yfinance` |
| 2 | `calculate_indicators(ticker, period)` | Percepção | RSI, MACD, **SMA 20/50**, **EMA 9/21**, Bandas de Bollinger e **Volume** (relativo + OBV) |
| 3 | `search_news(ticker)` | Percepção + Raciocínio | Notícias via RSS + **sentimento classificado por LLM**, score de impacto e **resumo dos eventos** |
| 4 | `generate_recommendation(analysis)` | Ação | Converte as observações em **COMPRAR / VENDER / AGUARDAR** com fatores explicáveis |
| 5 | `run_backtest_strategy(ticker, dias)` | Avaliação | Acurácia de tendência e retorno vs *Buy-and-Hold* |
| 6 | `plot_technical_chart(ticker, period)` | Ação | Gráfico técnico interativo em Plotly |

> **Nota sobre `calculate_indicators`:** no slide a assinatura é `calculate_indicators(data)`. Como ferramentas de agentes trafegam apenas tipos JSON (a LLM não consegue enviar um DataFrame), a ferramenta recebe o `ticker` e carrega internamente a mesma série OHLCV de `get_price_data`.


In [ ]:
# ---------- Utilitários compartilhados por todas as ferramentas ----------
EMPRESAS_B3 = {
    "VALE3": {"nome": "Vale", "termos": ["vale3", " vale ", "vale s.a", "mineradora vale", "minério de ferro"]},
    "PETR4": {"nome": "Petrobras", "termos": ["petr4", "petrobras"]},
    "BBAS3": {"nome": "Banco do Brasil", "termos": ["bbas3", "banco do brasil"]},
    "ITUB4": {"nome": "Itaú Unibanco", "termos": ["itub4", "itaú", "itau"]},
}


def limpar_ticker(ticker: str) -> str:
    """Remove sufixos e espaços, devolvendo o código da B3 em caixa alta (ex: 'petr4.sa' -> 'PETR4')."""
    return ticker.upper().replace(".SA", "").strip()


def normalizar_ticker(ticker: str) -> str:
    """Garante que o ticker possua a extensão .SA correspondente à B3 no Yahoo Finance."""
    t = ticker.upper().strip()
    if not t.endswith(".SA") and "." not in t:
        return f"{t}.SA"
    return t


def baixar_historico(ticker: str, period: str = "6mo") -> pd.DataFrame:
    """Baixa o histórico diário OHLCV do Yahoo Finance (retorna DataFrame vazio em caso de falha)."""
    try:
        return yf.Ticker(normalizar_ticker(ticker)).history(period=period)
    except Exception:
        return pd.DataFrame()


def get_price_data(ticker: str, period: str = "3mo") -> dict:
    """Coleta dados históricos de cotação e volume (OHLCV) de uma ação negociada na B3 via Yahoo Finance.

    Args:
        ticker: O código da ação na B3 (ex: 'VALE3', 'PETR4', 'BBAS3', 'ITUB4').
        period: Intervalo histórico a ser coletado (ex: '1mo', '3mo', '6mo', '1y').

    Returns:
        Um dicionário com o último preço, variação diária e no período, máxima, mínima e volume médio.
    """
    df = baixar_historico(ticker, period)

    if df.empty or len(df) < 5:
        return {
            "status": "erro",
            "mensagem": f"Não foi possível obter dados históricos para o ativo {ticker}."
        }

    ultimo_preco = float(df['Close'].iloc[-1])
    preco_anterior = float(df['Close'].iloc[-2])
    preco_inicio = float(df['Close'].iloc[0])

    return {
        "status": "sucesso",
        "ticker": limpar_ticker(ticker),
        "data_cotacao": df.index[-1].strftime('%Y-%m-%d'),
        "ultimo_preco": round(ultimo_preco, 2),
        "variacao_diaria_pct": round((ultimo_preco / preco_anterior - 1) * 100, 2),
        "variacao_periodo_pct": round((ultimo_preco / preco_inicio - 1) * 100, 2),
        "maxima_periodo": round(float(df['High'].max()), 2),
        "minima_periodo": round(float(df['Low'].min()), 2),
        "volume_medio": int(df['Volume'].mean()),
        "total_pregoes": len(df)
    }

# Teste unitário da Tool 1
teste_precos = get_price_data("PETR4", period="1mo")
print("Resultado get_price_data('PETR4', '1mo'):")
print(json.dumps(teste_precos, indent=2, ensure_ascii=False))


### 5.1. Indicadores Técnicos (RSI, MACD, SMA/EMA, Bollinger e Volume)

Todos os indicadores são calculados nativamente com `pandas`, sem dependências compiladas (compatível com Colab e Databricks Free):

| Indicador | Parâmetros | Leitura usada pelo agente |
|:--|:--|:--|
| **RSI** | 14 períodos | < 30 sobrevendido · > 70 sobrecomprado |
| **MACD** | EMA 12 − EMA 26, sinal EMA 9 | Linha acima do sinal = *bullish* |
| **SMA** | 20 e 50 | Preço > SMA20 > SMA50 = tendência de alta |
| **EMA** | 9 e 21 | EMA9 > EMA21 = momentum de curto prazo positivo |
| **Bollinger** | SMA20 ± 2σ | Preço fora das bandas = estiramento / possível reversão |
| **Volume** | Média 5d ÷ média 20d e OBV (10d) | Volume relativo ≥ 1,2 confirma o movimento; OBV indica fluxo comprador/vendedor |


In [ ]:
def calcular_series_indicadores(df: pd.DataFrame) -> pd.DataFrame:
    """Acrescenta ao DataFrame OHLCV as séries de todos os indicadores técnicos usados pelo agente."""
    df = df.copy()
    close = df['Close']
    delta = close.diff()

    # RSI (14)
    ganho = delta.clip(lower=0).rolling(window=14).mean()
    perda = (-delta.clip(upper=0)).rolling(window=14).mean()
    rs = ganho / perda.replace(0, np.nan)
    df['RSI'] = 100 - (100 / (1 + rs))

    # MACD (12, 26, 9)
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    df['MACD'] = ema12 - ema26
    df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD_Hist'] = df['MACD'] - df['MACD_Signal']

    # Médias Móveis Simples e Exponenciais
    df['SMA20'] = close.rolling(window=20).mean()
    df['SMA50'] = close.rolling(window=50).mean()
    df['EMA9'] = close.ewm(span=9, adjust=False).mean()
    df['EMA21'] = close.ewm(span=21, adjust=False).mean()

    # Bandas de Bollinger (20, 2σ)
    desvio20 = close.rolling(window=20).std()
    df['BB_Upper'] = df['SMA20'] + 2 * desvio20
    df['BB_Lower'] = df['SMA20'] - 2 * desvio20

    # Volume: média de 20 dias, volume relativo (5d / 20d) e On-Balance Volume
    df['Vol_SMA20'] = df['Volume'].rolling(window=20).mean()
    df['Vol_Relativo'] = df['Volume'].rolling(window=5).mean() / df['Vol_SMA20']
    df['OBV'] = (np.sign(delta).fillna(0) * df['Volume']).cumsum()
    df['OBV_Delta10'] = df['OBV'].diff(10)
    return df


COLUNAS_ESSENCIAIS = ['RSI', 'MACD', 'MACD_Signal', 'SMA20', 'SMA50', 'EMA9', 'EMA21',
                      'BB_Upper', 'BB_Lower', 'Vol_Relativo', 'OBV_Delta10']


def sinais_da_linha(linha: pd.Series) -> dict:
    """Traduz os valores numéricos de um pregão em sinais discretos interpretáveis."""
    preco = float(linha['Close'])
    if preco > linha['SMA20'] > linha['SMA50']:
        tendencia = "Alta"
    elif preco < linha['SMA20'] < linha['SMA50']:
        tendencia = "Baixa"
    else:
        tendencia = "Lateral"

    if preco > linha['BB_Upper']:
        posicao_bb = "acima"
    elif preco < linha['BB_Lower']:
        posicao_bb = "abaixo"
    else:
        posicao_bb = "dentro"

    return {
        "macd_signal": "bullish" if linha['MACD'] > linha['MACD_Signal'] else "bearish",
        "tendencia_medias": tendencia,
        "sinal_ema": "alta" if linha['EMA9'] > linha['EMA21'] else "baixa",
        "posicao_bollinger": posicao_bb,
        "volume_relativo": round(float(linha['Vol_Relativo']), 2),
        "obv_tendencia": "comprador" if linha['OBV_Delta10'] > 0 else "vendedor",
    }


def calculate_indicators(ticker: str, period: str = "6mo") -> dict:
    """Calcula os indicadores de análise técnica de uma ação da B3: RSI (14), MACD (12, 26, 9),
    Médias Móveis Simples (SMA 20 e 50) e Exponenciais (EMA 9 e 21), Bandas de Bollinger (20, 2σ)
    e indicadores de Volume (volume relativo 5d/20d e tendência do OBV).

    Args:
        ticker: O código da ação (ex: 'VALE3', 'PETR4', 'BBAS3', 'ITUB4').
        period: Período histórico usado no cálculo (mínimo '3mo'; padrão '6mo').

    Returns:
        Dicionário com os valores numéricos dos indicadores e a leitura técnica de cada um.
    """
    df = baixar_historico(ticker, period)
    if df.empty or len(df) < 60:
        return {
            "status": "erro",
            "mensagem": f"Histórico insuficiente para cálculo de indicadores em {ticker} (use period >= '3mo')."
        }

    df = calcular_series_indicadores(df).dropna(subset=COLUNAS_ESSENCIAIS)
    ultima = df.iloc[-1]
    sinais = sinais_da_linha(ultima)
    rsi_atual = float(ultima['RSI'])

    if rsi_atual > 70:
        rsi_diagnostico = "Sobrecomprado (alerta de possível correção)"
    elif rsi_atual < 30:
        rsi_diagnostico = "Sobrevendido (oportunidade de compra técnica)"
    else:
        rsi_diagnostico = "Neutro (equilíbrio entre compradores e vendedores)"

    vol_rel = sinais["volume_relativo"]
    if vol_rel >= 1.2:
        leitura_volume = "Volume acima da média: confirma a força do movimento atual"
    elif vol_rel <= 0.8:
        leitura_volume = "Volume abaixo da média: movimento com pouca convicção"
    else:
        leitura_volume = "Volume dentro da média histórica"

    return {
        "status": "sucesso",
        "ticker": limpar_ticker(ticker),
        "data_referencia": df.index[-1].strftime('%Y-%m-%d'),
        "ultimo_preco": round(float(ultima['Close']), 2),
        "rsi": round(rsi_atual, 2),
        "rsi_diagnostico": rsi_diagnostico,
        "macd": round(float(ultima['MACD']), 3),
        "macd_linha_sinal": round(float(ultima['MACD_Signal']), 3),
        "macd_histograma": round(float(ultima['MACD_Hist']), 3),
        "macd_signal": sinais["macd_signal"],
        "sma_20": round(float(ultima['SMA20']), 2),
        "sma_50": round(float(ultima['SMA50']), 2),
        "tendencia_medias": sinais["tendencia_medias"],
        "ema_9": round(float(ultima['EMA9']), 2),
        "ema_21": round(float(ultima['EMA21']), 2),
        "sinal_ema": sinais["sinal_ema"],
        "bollinger_superior": round(float(ultima['BB_Upper']), 2),
        "bollinger_inferior": round(float(ultima['BB_Lower']), 2),
        "posicao_bollinger": sinais["posicao_bollinger"],
        "volume_ultimo_pregao": int(ultima['Volume']),
        "volume_medio_20d": int(ultima['Vol_SMA20']),
        "volume_relativo": vol_rel,
        "obv_tendencia": sinais["obv_tendencia"],
        "leitura_volume": leitura_volume
    }

# Teste unitário da Tool 2
teste_indicadores = calculate_indicators("VALE3")
print("Resultado calculate_indicators('VALE3'):")
print(json.dumps(teste_indicadores, indent=2, ensure_ascii=False))


### 5.2. Notícias, Sentimento via LLM e Resumo de Eventos

Conforme o slide 5 do enunciado, a ferramenta `search_news`:
1. **Coleta** manchetes recentes via `feedparser` (Google News Brasil filtrado pelo ticker e pelo nome da empresa nos últimos 7 dias + feed do InfoMoney filtrado por palavras-chave).
2. **Classifica com LLM** cada manchete em *Positivo / Negativo / Neutro*, com score de −1,0 a +1,0 e um **peso de impacto** (0 a 1) sobre a ação.
3. Calcula o **score de impacto por ação** (média ponderada pelo impacto) e gera um **resumo automático dos principais eventos**.

Se não houver chave de LLM (ou a chamada falhar), a ferramenta usa um **léxico financeiro em português** e um resumo extrativo, mantendo o mesmo formato de saída. O campo `metodo_sentimento` informa qual caminho foi usado. Os resultados ficam em cache por 15 minutos para evitar *rate limits*.


In [ ]:
CACHE_NOTICIAS = {}
CACHE_TTL_SEG = 15 * 60
MODELO_SENTIMENTO_GEMINI = "gemini-2.5-flash"
MODELO_SENTIMENTO_OPENAI = "openai/gpt-4o-mini"

TERMOS_POSITIVOS = [
    "alta", "lucro", "supera", "recorde", "dividendo", "crescimento", "salto", "avanço",
    "valorização", "recomenda compra", "positivo", "otimismo", "dispara", "elevação",
    "forte", "expansão", "sobe", "eleva", "aprova", "upgrade"
]
TERMOS_NEGATIVOS = [
    "queda", "prejuízo", "crise", "recuo", "baixa", "corte", "risco", "despenca",
    "investigação", "desaceleração", "perda", "alerta", "negativo", "pessimismo",
    "pressão", "rebaixa", "multa", "cai", "recua", "downgrade"
]


def coletar_rss(url: str, limite: int) -> list:
    """Lê um feed RSS com timeout e devolve até `limite` entradas (lista vazia em caso de falha)."""
    try:
        resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=8)
        return feedparser.parse(resp.content).entries[:limite]
    except Exception:
        return []


def coletar_noticias(ticker: str, max_noticias: int) -> list:
    """Agrega manchetes do Google News (ticker e nome da empresa) e do InfoMoney, sem duplicatas."""
    info = EMPRESAS_B3.get(ticker, {"nome": ticker, "termos": [ticker.lower()]})
    noticias, vistos = [], set()

    def adicionar(titulo, data, fonte, link):
        chave = titulo.lower()[:80]
        if titulo and chave not in vistos:
            vistos.add(chave)
            noticias.append({"titulo": titulo, "data": data, "fonte": fonte, "link": link})

    for consulta in [f'"{ticker}"', f'"{info["nome"]}" ações']:
        url = ("https://news.google.com/rss/search?q=" + requests.utils.quote(consulta + " when:7d")
               + "&hl=pt-BR&gl=BR&ceid=BR:pt-419")
        for e in coletar_rss(url, max_noticias):
            fonte = e.get("source", {}).get("title", "Google News") if e.get("source") else "Google News"
            adicionar(e.get("title", "").strip(), e.get("published", ""), fonte, e.get("link", ""))

    for e in coletar_rss("https://www.infomoney.com.br/feed/", 60):
        titulo = e.get("title", "").strip()
        if any(t in f" {titulo.lower()} " for t in info["termos"]):
            adicionar(titulo, e.get("published", ""), "InfoMoney", e.get("link", ""))

    return noticias[:max_noticias]


def classificar_com_llm(ticker: str, noticias: list):
    """Classifica as manchetes com a LLM configurada. Retorna (dados_json, metodo) ou None se indisponível."""
    nome = EMPRESAS_B3.get(ticker, {}).get("nome", ticker)
    manchetes = "\n".join(f"{i + 1}. [{n['fonte']}] {n['titulo']}" for i, n in enumerate(noticias))
    prompt = (
        "Você é um analista de research sell-side especializado em ações brasileiras. "
        f"Avalie o efeito provável de cada manchete sobre o preço da ação {ticker} ({nome}) no curto prazo.\n"
        "Regras: sentimento em Positivo, Negativo ou Neutro; score de -1.0 (muito negativo) a +1.0 (muito positivo); "
        "impacto de 0.0 (irrelevante para esta ação) a 1.0 (impacto direto e material).\n"
        "Responda APENAS com JSON válido no formato: "
        '{"noticias": [{"id": 1, "sentimento": "Positivo", "score": 0.6, "impacto": 0.8, "motivo": "frase curta"}], '
        '"resumo_eventos": "2 a 3 frases resumindo os principais eventos e o que significam para a ação"}\n\n'
        "Manchetes:\n" + manchetes
    )

    try:
        if MODEL_PROVIDER == "gemini" and os.environ.get("GEMINI_API_KEY"):
            from google import genai
            from google.genai import types as genai_types
            cliente = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
            resp = cliente.models.generate_content(
                model=MODELO_SENTIMENTO_GEMINI,
                contents=prompt,
                config=genai_types.GenerateContentConfig(temperature=0.0, response_mime_type="application/json"),
            )
            texto, metodo = resp.text, f"LLM ({MODELO_SENTIMENTO_GEMINI})"
        elif os.environ.get("OPENAI_API_KEY"):
            import litellm
            resp = litellm.completion(
                model=MODELO_SENTIMENTO_OPENAI,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                response_format={"type": "json_object"},
            )
            texto, metodo = resp.choices[0].message.content, f"LLM ({MODELO_SENTIMENTO_OPENAI})"
        else:
            return None

        texto = texto.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        return json.loads(texto), metodo
    except Exception as e:
        print(f"ℹ️ Sentimento via LLM indisponível ({type(e).__name__}); usando léxico financeiro.")
        return None


def classificar_com_lexico(ticker: str, noticias: list) -> dict:
    """Fallback determinístico: classifica por contagem de termos e gera um resumo extrativo."""
    termos_empresa = EMPRESAS_B3.get(ticker, {}).get("termos", [ticker.lower()])
    itens = []
    for i, n in enumerate(noticias):
        txt = f" {n['titulo'].lower()} "
        pos = sum(1 for p in TERMOS_POSITIVOS if p in txt)
        neg = sum(1 for p in TERMOS_NEGATIVOS if p in txt)
        if pos > neg:
            score, rotulo = min(1.0, 0.25 + 0.25 * (pos - neg)), "Positivo"
        elif neg > pos:
            score, rotulo = -min(1.0, 0.25 + 0.25 * (neg - pos)), "Negativo"
        else:
            score, rotulo = 0.0, "Neutro"
        impacto = 1.0 if any(t in txt for t in termos_empresa) else 0.5
        itens.append({"id": i + 1, "sentimento": rotulo, "score": score, "impacto": impacto,
                      "motivo": f"{pos} termo(s) positivo(s) e {neg} negativo(s)"})

    relevantes = sorted(zip(noticias, itens), key=lambda x: x[1]["impacto"] * (abs(x[1]["score"]) + 0.1), reverse=True)
    resumo = "Resumo extrativo (sem LLM) das manchetes mais relevantes: " + "; ".join(n["titulo"] for n, _ in relevantes[:3]) + "."
    return {"noticias": itens, "resumo_eventos": resumo}


def search_news(ticker: str, max_noticias: int = 8) -> dict:
    """Busca notícias financeiras recentes da ação em feeds RSS (Google News Brasil e InfoMoney),
    classifica o sentimento de cada manchete com LLM (Positivo/Negativo/Neutro), calcula o score de
    impacto por ação e gera um resumo automático dos principais eventos.

    Args:
        ticker: O código da ação (ex: 'VALE3', 'PETR4', 'BBAS3', 'ITUB4').
        max_noticias: Quantidade máxima de manchetes a analisar (padrão 8).

    Returns:
        Dicionário com news_sentiment (-1.0 a +1.0, ponderado pelo impacto), classificação geral,
        score de impacto médio, resumo dos eventos e a lista de manchetes classificadas.
    """
    t = limpar_ticker(ticker)
    chave_cache = (t, max_noticias)
    if chave_cache in CACHE_NOTICIAS and time.time() - CACHE_NOTICIAS[chave_cache][0] < CACHE_TTL_SEG:
        return CACHE_NOTICIAS[chave_cache][1]

    noticias = coletar_noticias(t, max_noticias)
    if not noticias:
        return {
            "status": "sucesso",
            "ticker": t,
            "total_noticias": 0,
            "news_sentiment": 0.0,
            "classificacao_sentimento": "Neutro",
            "score_impacto": 0.0,
            "resumo_eventos": "Nenhuma notícia recente encontrada nos feeds consultados.",
            "metodo_sentimento": "sem dados",
            "noticias_analisadas": []
        }

    resultado_llm = classificar_com_llm(t, noticias)
    if resultado_llm:
        dados, metodo = resultado_llm
    else:
        dados, metodo = classificar_com_lexico(t, noticias), "Léxico financeiro (fallback)"

    por_id = {int(item.get("id", 0)): item for item in dados.get("noticias", []) if str(item.get("id", "")).isdigit()}
    analisadas, soma_pesos, soma_ponderada = [], 0.0, 0.0
    for i, n in enumerate(noticias):
        item = por_id.get(i + 1, {})
        score = max(-1.0, min(1.0, float(item.get("score", 0.0))))
        impacto = max(0.0, min(1.0, float(item.get("impacto", 0.5))))
        soma_pesos += impacto
        soma_ponderada += score * impacto
        analisadas.append({
            "titulo": n["titulo"],
            "fonte": n["fonte"],
            "data": n["data"],
            "sentimento": item.get("sentimento", "Neutro"),
            "score": round(score, 2),
            "impacto": round(impacto, 2),
            "motivo": item.get("motivo", "")
        })

    news_sentiment = round(soma_ponderada / soma_pesos, 2) if soma_pesos > 0 else 0.0
    if news_sentiment >= 0.25:
        classificacao = "Positivo"
    elif news_sentiment <= -0.25:
        classificacao = "Negativo"
    else:
        classificacao = "Neutro"

    resultado = {
        "status": "sucesso",
        "ticker": t,
        "total_noticias": len(noticias),
        "news_sentiment": news_sentiment,
        "classificacao_sentimento": classificacao,
        "score_impacto": round(soma_pesos / len(noticias), 2),
        "resumo_eventos": dados.get("resumo_eventos", ""),
        "metodo_sentimento": metodo,
        "noticias_analisadas": sorted(analisadas, key=lambda x: x["impacto"], reverse=True)[:6]
    }
    CACHE_NOTICIAS[chave_cache] = (time.time(), resultado)
    return resultado

# Teste unitário da Tool 3
teste_noticias = search_news("PETR4")
print("Resultado search_news('PETR4'):")
print(json.dumps(teste_noticias, indent=2, ensure_ascii=False))


## 6. Módulo de Ação: Recomendação, Backtest e Visualização

### 6.1. `generate_recommendation(analysis)` — Síntese Decisória Explicável

A LLM reúne as observações das ferramentas de percepção e as envia como argumentos desta ferramenta, que aplica um **modelo de pontuação multifator transparente**. Cada fator gera pontos positivos (compra) ou negativos (venda), e a soma define a recomendação:

| Fator | Regra | Pontos |
|:--|:--|:-:|
| RSI | < 30 / 30–45 / 45–60 / 60–70 / > 70 | +2 / +0,5 / 0 / −0,5 / −2 |
| MACD | *bullish* / *bearish* | +1 / −1 |
| Tendência SMA 20/50 | Alta / Lateral / Baixa | +1 / 0 / −1 |
| Cruzamento EMA 9/21 | alta / baixa | +1 / −1 |
| Bollinger | abaixo da banda / dentro / acima | +1 / 0 / −1 |
| Volume relativo | ≥ 1,2 reforça a direção técnica | ±0,5 |
| Sentimento das notícias | `news_sentiment` × 2 | −2 a +2 |

**Decisão:** soma ≥ +2 → `COMPRAR` · soma ≤ −2 → `VENDER` · caso contrário → `AGUARDAR`. Todas as recomendações ficam guardadas em `REGISTRO_RECOMENDACOES` para auditoria.


In [ ]:
REGISTRO_RECOMENDACOES = []
LIMIAR_DECISAO = 2.0
PONTUACAO_MAXIMA = 8.5


def motor_decisao(rsi: float, macd_signal: str, tendencia_medias: str, sinal_ema: str,
                  posicao_bollinger: str, volume_relativo: float, news_sentiment: float) -> dict:
    """Modelo de pontuação multifator compartilhado pela recomendação ao vivo e pelo backtest."""
    fatores = []

    if rsi < 30:
        fatores.append(("RSI", f"{rsi:.1f} sobrevendido", 2.0))
    elif rsi < 45:
        fatores.append(("RSI", f"{rsi:.1f} em zona baixa, com espaço para alta", 0.5))
    elif rsi <= 60:
        fatores.append(("RSI", f"{rsi:.1f} neutro", 0.0))
    elif rsi <= 70:
        fatores.append(("RSI", f"{rsi:.1f} próximo da sobrecompra", -0.5))
    else:
        fatores.append(("RSI", f"{rsi:.1f} sobrecomprado", -2.0))

    macd = str(macd_signal).lower()
    fatores.append(("MACD", macd, 1.0 if macd == "bullish" else -1.0))

    tend = str(tendencia_medias).capitalize()
    fatores.append(("Tendência SMA 20/50", tend, {"Alta": 1.0, "Baixa": -1.0}.get(tend, 0.0)))

    ema = str(sinal_ema).lower()
    fatores.append(("Cruzamento EMA 9/21", ema, {"alta": 1.0, "baixa": -1.0}.get(ema, 0.0)))

    bb = str(posicao_bollinger).lower()
    leitura_bb = {"abaixo": "preço abaixo da banda inferior", "acima": "preço acima da banda superior"}.get(bb, "preço dentro das bandas")
    fatores.append(("Bandas de Bollinger", leitura_bb, {"abaixo": 1.0, "acima": -1.0}.get(bb, 0.0)))

    soma_tecnica = sum(p for _, _, p in fatores)
    if volume_relativo >= 1.2 and soma_tecnica != 0:
        fatores.append(("Volume", f"{volume_relativo:.2f}x a média, confirma o movimento", 0.5 if soma_tecnica > 0 else -0.5))
    else:
        fatores.append(("Volume", f"{volume_relativo:.2f}x a média, sem confirmação", 0.0))

    sent = max(-1.0, min(1.0, float(news_sentiment)))
    fatores.append(("Sentimento das notícias", f"{sent:+.2f}", round(sent * 2, 2)))

    total = round(sum(p for _, _, p in fatores), 2)
    if total >= LIMIAR_DECISAO:
        recomendacao = "COMPRAR"
    elif total <= -LIMIAR_DECISAO:
        recomendacao = "VENDER"
    else:
        recomendacao = "AGUARDAR"

    if recomendacao == "AGUARDAR":
        confianca = 0.5 + 0.3 * (1 - abs(total) / LIMIAR_DECISAO)
    else:
        confianca = 0.5 + 0.45 * min(1.0, abs(total) / PONTUACAO_MAXIMA * 2)

    return {
        "recommendation": recomendacao,
        "score_total": total,
        "confidence_score": round(min(confianca, 0.95), 2),
        "fatores": [{"fator": f, "leitura": l, "pontos": p} for f, l, p in fatores]
    }


def generate_recommendation(ticker: str, close: float, rsi: float, macd_signal: str, news_sentiment: float,
                            tendencia_medias: str = "Lateral", sinal_ema: str = "neutro",
                            posicao_bollinger: str = "dentro", volume_relativo: float = 1.0,
                            data_referencia: str = "") -> dict:
    """Gera a recomendação final (COMPRAR, VENDER ou AGUARDAR) a partir da análise consolidada.
    Use SEMPRE depois de calculate_indicators e search_news, repassando os valores observados.

    Args:
        ticker: Código da ação (ex: 'PETR4').
        close: Último preço de fechamento (campo 'ultimo_preco' de calculate_indicators).
        rsi: Valor do RSI (14) retornado por calculate_indicators.
        macd_signal: 'bullish' ou 'bearish' (campo 'macd_signal').
        news_sentiment: Score de sentimento de -1.0 a +1.0 retornado por search_news.
        tendencia_medias: 'Alta', 'Baixa' ou 'Lateral' (campo 'tendencia_medias').
        sinal_ema: 'alta' ou 'baixa' (campo 'sinal_ema', cruzamento EMA 9/21).
        posicao_bollinger: 'acima', 'abaixo' ou 'dentro' (campo 'posicao_bollinger').
        volume_relativo: Volume relativo 5d/20d (campo 'volume_relativo').
        data_referencia: Data da análise no formato YYYY-MM-DD (opcional; padrão = hoje).

    Returns:
        Recomendação, score de confiança, contribuição de cada fator, justificativa em linguagem
        natural e o registro canônico no formato oficial da QuantumFinance.
    """
    t = limpar_ticker(ticker)
    decisao = motor_decisao(float(rsi), macd_signal, tendencia_medias, sinal_ema,
                            posicao_bollinger, float(volume_relativo), float(news_sentiment))

    a_favor = [f for f in decisao["fatores"] if f["pontos"] > 0]
    contra = [f for f in decisao["fatores"] if f["pontos"] < 0]
    texto_favor = ", ".join(f"{f['fator']} ({f['leitura']})" for f in a_favor) or "nenhum"
    texto_contra = ", ".join(f"{f['fator']} ({f['leitura']})" for f in contra) or "nenhum"
    justificativa = (
        f"Pontuação total {decisao['score_total']:+.2f} (limiar ±{LIMIAR_DECISAO:.0f}). "
        f"Fatores de alta: {texto_favor}. Fatores de baixa: {texto_contra}."
    )

    registro = {
        "ticker": t,
        "date": data_referencia or datetime.today().strftime('%Y-%m-%d'),
        "close": round(float(close), 2),
        "rsi": round(float(rsi), 2),
        "macd_signal": str(macd_signal).lower(),
        "news_sentiment": round(float(news_sentiment), 2),
        "recommendation": decisao["recommendation"]
    }
    REGISTRO_RECOMENDACOES.append({**registro, "confidence_score": decisao["confidence_score"],
                                   "score_total": decisao["score_total"],
                                   "registrado_em": datetime.now().strftime('%Y-%m-%d %H:%M:%S')})

    return {
        "status": "sucesso",
        "recommendation": decisao["recommendation"],
        "confidence_score": decisao["confidence_score"],
        "score_total": decisao["score_total"],
        "fatores": decisao["fatores"],
        "rationale": justificativa,
        "registro_oficial": registro
    }

# Teste unitário da Tool 4 (encadeando as observações das Tools 2 e 3)
if teste_indicadores.get("status") == "sucesso":
    teste_rec = generate_recommendation(
        ticker="VALE3",
        close=teste_indicadores["ultimo_preco"],
        rsi=teste_indicadores["rsi"],
        macd_signal=teste_indicadores["macd_signal"],
        news_sentiment=search_news("VALE3")["news_sentiment"],
        tendencia_medias=teste_indicadores["tendencia_medias"],
        sinal_ema=teste_indicadores["sinal_ema"],
        posicao_bollinger=teste_indicadores["posicao_bollinger"],
        volume_relativo=teste_indicadores["volume_relativo"],
        data_referencia=teste_indicadores["data_referencia"],
    )
    print("Resultado generate_recommendation('VALE3', ...):")
    print(json.dumps(teste_rec, indent=2, ensure_ascii=False))


### 6.2. `run_backtest_strategy` — Acurácia de Tendência e Backtest

O backtest aplica **o mesmo `motor_decisao`** usado na recomendação ao vivo, pregão a pregão, sem olhar dados futuros (a decisão do dia *t* vale para o retorno de *t* → *t+1*):
- **Acurácia de tendência:** entre os sinais `COMPRAR`/`VENDER`, percentual em que o preço andou na direção indicada nos *N* pregões seguintes (padrão: 5).
- **Retorno da estratégia vs Buy-and-Hold:** comprado após `COMPRAR`, fora do mercado após `VENDER`, mantém a posição em `AGUARDAR`.

> **Transparência metodológica:** não existe arquivo histórico gratuito de manchetes por pregão, então o backtest avalia o **componente técnico** do modelo (sentimento = 0). O sentimento das notícias entra apenas nas recomendações ao vivo.


In [ ]:
def run_backtest_strategy(ticker: str, dias_teste: int = 120, horizonte_dias: int = 5) -> dict:
    """Simula as recomendações do agente nos últimos N pregões e compara com a estratégia Buy-and-Hold.
    Mede a acurácia de tendência (acerto de direção) e o retorno financeiro acumulado.

    Args:
        ticker: O código da ação (ex: 'VALE3', 'PETR4', 'BBAS3', 'ITUB4').
        dias_teste: Quantidade de pregões do período de teste (padrão 120, cerca de 6 meses).
        horizonte_dias: Pregões à frente usados para verificar o acerto de tendência (padrão 5).

    Returns:
        Retorno acumulado do agente e do Buy-and-Hold, alpha, acurácia de tendência e contagem de sinais.
    """
    df = baixar_historico(ticker, "2y")
    if df.empty or len(df) < dias_teste + 60:
        return {
            "status": "erro",
            "mensagem": f"Histórico insuficiente para backtest de {dias_teste} pregões em {ticker}."
        }

    teste = calcular_series_indicadores(df).dropna(subset=COLUNAS_ESSENCIAIS).tail(dias_teste)
    precos = teste['Close'].values
    n = len(teste)

    posicao, retornos, operacoes = 0, [], 0
    acertos, sinais_avaliados = 0, 0
    contagem = {"COMPRAR": 0, "VENDER": 0, "AGUARDAR": 0}

    for i in range(n - 1):
        linha = teste.iloc[i]
        s = sinais_da_linha(linha)
        rec = motor_decisao(float(linha['RSI']), s["macd_signal"], s["tendencia_medias"], s["sinal_ema"],
                            s["posicao_bollinger"], s["volume_relativo"], 0.0)["recommendation"]
        contagem[rec] += 1

        # Acurácia de tendência no horizonte definido
        if rec != "AGUARDAR" and i + horizonte_dias < n:
            retorno_futuro = precos[i + horizonte_dias] / precos[i] - 1
            sinais_avaliados += 1
            if (rec == "COMPRAR" and retorno_futuro > 0) or (rec == "VENDER" and retorno_futuro < 0):
                acertos += 1

        # Atualiza a posição e aplica o retorno do pregão seguinte
        nova_posicao = 1 if rec == "COMPRAR" else (0 if rec == "VENDER" else posicao)
        if nova_posicao == 1 and posicao == 0:
            operacoes += 1
        posicao = nova_posicao
        retornos.append((precos[i + 1] / precos[i] - 1) if posicao == 1 else 0.0)

    retorno_agente = (float(np.prod(1 + np.array(retornos))) - 1) * 100
    retorno_bh = (precos[-1] / precos[0] - 1) * 100
    acuracia = (acertos / sinais_avaliados * 100) if sinais_avaliados else None

    return {
        "status": "sucesso",
        "ticker": limpar_ticker(ticker),
        "periodo": f"{teste.index[0].strftime('%Y-%m-%d')} a {teste.index[-1].strftime('%Y-%m-%d')}",
        "dias_avaliados": n,
        "retorno_agente_pct": round(retorno_agente, 2),
        "retorno_buy_and_hold_pct": round(float(retorno_bh), 2),
        "alpha_gerado_pct": round(retorno_agente - float(retorno_bh), 2),
        "acuracia_tendencia_pct": round(acuracia, 2) if acuracia is not None else None,
        "horizonte_acuracia_dias": horizonte_dias,
        "sinais_avaliados": sinais_avaliados,
        "distribuicao_sinais": contagem,
        "total_operacoes": operacoes,
        "observacao": "Backtest do componente técnico (sentimento histórico não disponível)."
    }

# Teste unitário da Tool 5
teste_backtest = run_backtest_strategy("PETR4")
print("Resultado run_backtest_strategy('PETR4'):")
print(json.dumps(teste_backtest, indent=2, ensure_ascii=False))


### 6.3. `plot_technical_chart` — Visualização como Ferramenta do Agente

Atendendo ao slide 11 (*"Plotly / Matplotlib – Como Tool do Agente"*), o gráfico é uma ferramenta que a LLM pode acionar sozinha quando o usuário pede algo como *"mostre o gráfico de VALE3"*. A ferramenta:
- monta um painel com **Candlestick + SMA 20/50 + EMA 9/21 + Bollinger**, **Volume**, **MACD** e **RSI**;
- salva o gráfico interativo em `graficos/<TICKER>_analise_tecnica.html`;
- devolve ao agente uma **leitura textual** do gráfico, para que ele comente o que o usuário está vendo;
- disponibiliza a figura na variável `ULTIMO_GRAFICO`, exibida no notebook e no painel do Gradio.


In [ ]:
PASTA_GRAFICOS = "graficos"
ULTIMO_GRAFICO = None


def criar_grafico_tecnico(ticker: str, period: str = "6mo"):
    """Monta a figura Plotly com 4 painéis (preço, volume, MACD e RSI). Retorna (fig, df_indicadores)."""
    df = baixar_historico(ticker, period)
    if df.empty or len(df) < 30:
        return None, None
    df = calcular_series_indicadores(df)
    t = limpar_ticker(ticker)

    fig = make_subplots(
        rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.03,
        row_heights=[0.52, 0.14, 0.17, 0.17],
        subplot_titles=[f"{t} — Preço, Médias e Bollinger ({period})", "Volume", "MACD (12, 26, 9)", "RSI (14)"]
    )

    fig.add_trace(go.Candlestick(x=df.index, open=df['Open'], high=df['High'], low=df['Low'],
                                 close=df['Close'], name="Cotações"), row=1, col=1)
    for coluna, cor, estilo, nome in [
        ('SMA20', '#f5a623', 'solid', 'SMA 20'), ('SMA50', '#4a90e2', 'solid', 'SMA 50'),
        ('EMA9', '#7ed321', 'dot', 'EMA 9'), ('EMA21', '#bd10e0', 'dot', 'EMA 21'),
        ('BB_Upper', '#9b9b9b', 'dash', 'Bollinger Sup.'), ('BB_Lower', '#9b9b9b', 'dash', 'Bollinger Inf.'),
    ]:
        fig.add_trace(go.Scatter(x=df.index, y=df[coluna], name=nome,
                                 line=dict(color=cor, width=1.4, dash=estilo)), row=1, col=1)

    cores_volume = np.where(df['Close'] >= df['Open'], '#26a69a', '#ef5350')
    fig.add_trace(go.Bar(x=df.index, y=df['Volume'], marker_color=cores_volume, name="Volume", showlegend=False), row=2, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['Vol_SMA20'], name="Volume méd. 20",
                             line=dict(color='#f5a623', width=1.2)), row=2, col=1)

    cores_hist = np.where(df['MACD_Hist'] >= 0, '#26a69a', '#ef5350')
    fig.add_trace(go.Bar(x=df.index, y=df['MACD_Hist'], marker_color=cores_hist, name="Histograma", showlegend=False), row=3, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['MACD'], name="MACD", line=dict(color='#4a90e2', width=1.4)), row=3, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['MACD_Signal'], name="Sinal", line=dict(color='#f5a623', width=1.4)), row=3, col=1)

    fig.add_trace(go.Scatter(x=df.index, y=df['RSI'], name="RSI", line=dict(color='#bd10e0', width=1.6)), row=4, col=1)
    fig.add_hline(y=70, line=dict(color='#ef5350', dash='dot'), row=4, col=1)
    fig.add_hline(y=30, line=dict(color='#26a69a', dash='dot'), row=4, col=1)

    fig.update_layout(
        template="plotly_dark",
        title_text=f"Análise Técnica — QuantumFinance | {t}",
        xaxis_rangeslider_visible=False,
        height=900,
        legend=dict(orientation="h", y=1.04, x=0)
    )
    return fig, df


def plot_technical_chart(ticker: str, period: str = "6mo") -> dict:
    """Gera o gráfico técnico interativo (Plotly) da ação com Candlestick, SMA 20/50, EMA 9/21,
    Bandas de Bollinger, Volume, MACD e RSI. Use quando o usuário pedir um gráfico ou visualização.

    Args:
        ticker: O código da ação (ex: 'VALE3', 'PETR4', 'BBAS3', 'ITUB4').
        period: Período exibido no gráfico (ex: '3mo', '6mo', '1y'; padrão '6mo').

    Returns:
        Caminho do arquivo HTML gerado, painéis exibidos e uma leitura textual do gráfico.
    """
    global ULTIMO_GRAFICO
    fig, df = criar_grafico_tecnico(ticker, period)
    if fig is None:
        return {"status": "erro", "mensagem": f"Não foi possível gerar o gráfico de {ticker}."}

    t = limpar_ticker(ticker)
    os.makedirs(PASTA_GRAFICOS, exist_ok=True)
    caminho = os.path.join(PASTA_GRAFICOS, f"{t}_analise_tecnica.html")
    fig.write_html(caminho, include_plotlyjs="cdn")
    ULTIMO_GRAFICO = fig

    validas = df.dropna(subset=COLUNAS_ESSENCIAIS)
    leitura = "Dados insuficientes para leitura completa dos indicadores."
    if not validas.empty:
        ultima = validas.iloc[-1]
        s = sinais_da_linha(ultima)
        leitura = (
            f"Preço R$ {ultima['Close']:.2f} {s['posicao_bollinger']} das Bandas de Bollinger "
            f"(R$ {ultima['BB_Lower']:.2f} a R$ {ultima['BB_Upper']:.2f}); tendência das SMAs: {s['tendencia_medias']}; "
            f"EMA 9 {'acima' if s['sinal_ema'] == 'alta' else 'abaixo'} da EMA 21; MACD {s['macd_signal']}; "
            f"RSI {ultima['RSI']:.1f}; volume {s['volume_relativo']:.2f}x a média de 20 dias."
        )

    return {
        "status": "sucesso",
        "ticker": t,
        "arquivo_html": caminho,
        "paineis": ["Candlestick + SMA 20/50 + EMA 9/21 + Bollinger", "Volume", "MACD", "RSI"],
        "leitura_visual": leitura,
        "mensagem": "Gráfico gerado e exibido para o usuário."
    }

# Teste unitário da Tool 6
teste_grafico = plot_technical_chart("PETR4")
print(json.dumps(teste_grafico, indent=2, ensure_ascii=False))
if ULTIMO_GRAFICO is not None:
    ULTIMO_GRAFICO.show()


## 7. Construção do AI Agent com Google ADK & Resiliência Operacional

Nesta etapa, configuramos o agente utilizando as classes oficiais do **Google ADK**:
- `google.adk.agents.Agent`
- `google.adk.runners.Runner`
- `google.adk.sessions.InMemorySessionService`

O agente recebe uma **instrução de sistema (System Prompt)** rigorosa:
- Seguir o ciclo **Thought → Action → Observation** (ReAct) antes de responder.
- Coletar evidências com `get_price_data`, `calculate_indicators` e `search_news`.
- Decidir **somente** por meio de `generate_recommendation`, repassando os valores observados (a decisão fica auditável e reproduzível).
- Usar `plot_technical_chart` e `run_backtest_strategy` quando o usuário pedir gráfico ou validação histórica.

**Arquitetura Dual (Online LLM + Motor Resiliente Determinístico)**:
A função `perguntar_agente` registra cada *Action* executada pelo agente e implementa um mecanismo de contingência automático: caso a API da LLM atinja *rate limits* ou o usuário execute sem chave (como no Databricks Free), o motor analítico executa o mesmo ciclo ReAct com as mesmas ferramentas e dados reais.


In [ ]:
SYSTEM_INSTRUCTION = """Você é o QuantumAdvisor, um AI Agent autônomo e sênior em recomendação de investimentos da QuantumFinance, especializado em ações da B3 (VALE3, PETR4, BBAS3, ITUB4).

Seu papel é emitir recomendações autônomas, fundamentadas e explicáveis utilizando a metodologia ReAct (Reasoning + Acting).

FERRAMENTAS DISPONÍVEIS:
- get_price_data(ticker, period): preços e volumes recentes.
- calculate_indicators(ticker, period): RSI, MACD, SMA 20/50, EMA 9/21, Bandas de Bollinger e Volume.
- search_news(ticker): notícias recentes, sentimento (-1 a +1), score de impacto e resumo dos eventos.
- generate_recommendation(...): ÚNICA forma válida de decidir entre COMPRAR, VENDER ou AGUARDAR.
- run_backtest_strategy(ticker, dias_teste): acurácia de tendência e retorno vs Buy-and-Hold.
- plot_technical_chart(ticker, period): gráfico técnico interativo exibido ao usuário.

DIRETRIZES DE OPERAÇÃO:
1. Para qualquer pedido de análise ou recomendação de uma ação:
   a) Thought: explique quais evidências você precisa.
   b) Action: chame get_price_data, calculate_indicators e search_news para o ticker.
   c) Observation: interprete cada retorno (indicadores técnicos, volume, sentimento e eventos).
   d) Action: chame generate_recommendation repassando EXATAMENTE os valores observados
      (ultimo_preco, rsi, macd_signal, news_sentiment, tendencia_medias, sinal_ema, posicao_bollinger,
      volume_relativo, data_referencia). Nunca invente a recomendação sem essa ferramenta.
2. Se o usuário pedir gráfico/visualização, chame plot_technical_chart e comente a 'leitura_visual'.
3. Se o usuário pedir desempenho histórico, validação ou backtest, chame run_backtest_strategy.
4. Para perguntas como "Por que você recomendou...?", baseie a explicação nos 'fatores' e no 'rationale'
   retornados por generate_recommendation (chame as ferramentas novamente se necessário).

FORMATO DA RESPOSTA (em português):
- Resumo executivo com a recomendação clara (COMPRAR, VENDER ou AGUARDAR) e o confidence_score.
- Raciocínio passo a passo (Thought / Action / Observation) de forma explicável.
- Tabela com os indicadores observados e a contribuição de cada fator.
- Resumo dos eventos noticiados que influenciaram o sentimento.
- O objeto JSON final no formato oficial da QuantumFinance (campo 'registro_oficial'):
  { "ticker": "...", "date": "YYYY-MM-DD", "close": ..., "rsi": ..., "macd_signal": "...", "news_sentiment": ..., "recommendation": "COMPRAR/VENDER/AGUARDAR" }
"""

FERRAMENTAS_AGENTE = [
    get_price_data,
    calculate_indicators,
    search_news,
    generate_recommendation,
    run_backtest_strategy,
    plot_technical_chart,
]


In [ ]:
# Inicialização do Agente e Sessão usando Google ADK
APP_NAME = "quantum_finance_app"
USER_ID = "analista_investimentos"

try:
    from google.adk.agents import Agent
    from google.adk.models.lite_llm import LiteLlm
    from google.adk.runners import Runner
    from google.adk.sessions import InMemorySessionService
    from google.genai import types

    # Definição do modelo conforme o provedor configurado
    if MODEL_PROVIDER == "gemini" and os.environ.get("GEMINI_API_KEY"):
        modelo = "gemini-2.5-flash"
    elif os.environ.get("OPENAI_API_KEY"):
        modelo = LiteLlm(model="openai/gpt-4o-mini")
    else:
        # Fallback local se nenhuma chave estiver configurada
        modelo = None

    if modelo is not None:
        agente_quantum = Agent(
            name="quantum_finance_advisor",
            model=modelo,
            instruction=SYSTEM_INSTRUCTION,
            tools=FERRAMENTAS_AGENTE
        )

        session_service = InMemorySessionService()
        runner = Runner(
            agent=agente_quantum,
            app_name=APP_NAME,
            session_service=session_service
        )
        ADK_DISPONIVEL = True
        print(f"🤖 Google ADK Agent inicializado com {len(FERRAMENTAS_AGENTE)} ferramentas!")
    else:
        ADK_DISPONIVEL = False
        print("ℹ️ Google ADK configurado no modo de simulação analítica.")
except Exception as e:
    ADK_DISPONIVEL = False
    print(f"ℹ️ Google ADK executará via motor analítico determinístico: {e}")


In [ ]:
async def garantir_sessao(session_id: str):
    """Cria a sessão no InMemorySessionService caso ainda não exista (compatível com APIs sync e async do ADK)."""
    existente = session_service.get_session(app_name=APP_NAME, user_id=USER_ID, session_id=session_id)
    if inspect.isawaitable(existente):
        existente = await existente
    if existente is None:
        criada = session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=session_id)
        if inspect.isawaitable(criada):
            await criada


ALIASES_TICKERS = {
    "VALE3": ["VALE3", "VALE"],
    "PETR4": ["PETR4", "PETROBRAS"],
    "BBAS3": ["BBAS3", "BANCO DO BRASIL"],
    "ITUB4": ["ITUB4", "ITAÚ", "ITAU"],
}


def identificar_tickers(mensagem: str) -> list:
    """Detecta os tickers (ou nomes das empresas) citados na mensagem do usuário."""
    msg = mensagem.upper()
    return [t for t, apelidos in ALIASES_TICKERS.items()
            if any(re.search(r"\b" + re.escape(a) + r"\b", msg) for a in apelidos)]


def analisar_ticker_offline(t: str, incluir_backtest: bool, incluir_grafico: bool) -> tuple:
    """Executa o ciclo ReAct completo para um ticker com as mesmas ferramentas do agente. Retorna (markdown, registro)."""
    passos = []
    passos.append(f"**Thought 1:** Preciso do preço atual e do comportamento recente de {t}.")
    precos = get_price_data(t, "3mo")
    passos.append(f"**Action 1:** `get_price_data('{t}', '3mo')`")
    if precos.get("status") != "sucesso":
        return f"### {t}\n⚠️ {precos.get('mensagem')}", None
    passos.append(f"**Observation 1:** Fechamento de R$ {precos['ultimo_preco']:.2f} ({precos['variacao_diaria_pct']:+.2f}% no dia; "
                  f"{precos['variacao_periodo_pct']:+.2f}% em 3 meses).")

    passos.append("**Thought 2:** Agora avalio tendência, momentum, volatilidade e volume.")
    ind = calculate_indicators(t)
    passos.append(f"**Action 2:** `calculate_indicators('{t}', '6mo')`")
    if ind.get("status") != "sucesso":
        return f"### {t}\n⚠️ {ind.get('mensagem')}", None
    passos.append(f"**Observation 2:** RSI {ind['rsi']:.1f} ({ind['rsi_diagnostico']}); MACD {ind['macd_signal']} "
                  f"(histograma {ind['macd_histograma']}); SMAs em tendência {ind['tendencia_medias']}; EMA 9/21 em {ind['sinal_ema']}; "
                  f"preço {ind['posicao_bollinger']} das Bandas de Bollinger; {ind['leitura_volume'].lower()} ({ind['volume_relativo']:.2f}x).")

    passos.append("**Thought 3:** Preciso verificar se as notícias confirmam ou contradizem o quadro técnico.")
    news = search_news(t)
    passos.append(f"**Action 3:** `search_news('{t}')`")
    passos.append(f"**Observation 3:** {news['total_noticias']} manchetes; sentimento {news['news_sentiment']:+.2f} "
                  f"({news['classificacao_sentimento']}, via {news['metodo_sentimento']}). Eventos: {news['resumo_eventos']}")

    passos.append("**Thought 4:** Tenho evidências suficientes; consolido a decisão no modelo multifator.")
    rec = generate_recommendation(
        ticker=t, close=ind["ultimo_preco"], rsi=ind["rsi"], macd_signal=ind["macd_signal"],
        news_sentiment=news["news_sentiment"], tendencia_medias=ind["tendencia_medias"],
        sinal_ema=ind["sinal_ema"], posicao_bollinger=ind["posicao_bollinger"],
        volume_relativo=ind["volume_relativo"], data_referencia=ind["data_referencia"],
    )
    passos.append("**Action 4:** `generate_recommendation(...)` com os valores observados acima")
    passos.append(f"**Observation 4:** {rec['recommendation']} (confiança {rec['confidence_score']:.0%}). {rec['rationale']}")

    extras = []
    if incluir_grafico:
        graf = plot_technical_chart(t)
        passos.append(f"**Action 5:** `plot_technical_chart('{t}')` → {graf.get('leitura_visual', graf.get('mensagem'))}")
    if incluir_backtest:
        bt = run_backtest_strategy(t)
        passos.append(f"**Action 6:** `run_backtest_strategy('{t}')`")
        if bt.get("status") == "sucesso":
            acc = f"{bt['acuracia_tendencia_pct']:.1f}%" if bt['acuracia_tendencia_pct'] is not None else "n/d"
            extras.append(f"**Backtest ({bt['periodo']}):** agente {bt['retorno_agente_pct']:+.2f}% vs Buy & Hold "
                          f"{bt['retorno_buy_and_hold_pct']:+.2f}% · acurácia de tendência {acc} em {bt['sinais_avaliados']} sinais.")

    tabela = ["| Fator | Leitura | Pontos |", "|:--|:--|:-:|"]
    tabela += [f"| {f['fator']} | {f['leitura']} | {f['pontos']:+.2f} |" for f in rec["fatores"]]
    tabela.append(f"| **Total** | limiar ±{LIMIAR_DECISAO:.0f} | **{rec['score_total']:+.2f}** |")

    noticias_md = [f"- [{n['sentimento']} {n['score']:+.2f} · impacto {n['impacto']:.1f}] {n['titulo']} — *{n['fonte']}*"
                   for n in news["noticias_analisadas"][:4]] or ["- Nenhuma manchete recente encontrada."]

    partes = [
        f"### 📊 Relatório Executivo QuantumFinance: {t}",
        f"**Recomendação Oficial:** `{rec['recommendation']}` · **Confiança:** {rec['confidence_score']:.0%} · "
        f"**Fechamento:** R$ {ind['ultimo_preco']:.2f} ({ind['data_referencia']})",
        "#### 🧠 Raciocínio do Agente (Thought → Action → Observation)",
        "\n\n".join(passos),
        "#### ⚖️ Contribuição de Cada Fator",
        "\n".join(tabela),
        "#### 📰 Principais Manchetes Classificadas",
        "\n".join(noticias_md),
        *extras,
        "#### 📋 Estrutura Canônica de Saída (Slide 7)",
        "```json\n" + json.dumps(rec["registro_oficial"], indent=2, ensure_ascii=False) + "\n```",
    ]
    return "\n\n".join(partes), rec["registro_oficial"]


def motor_analitico_offline(mensagem: str) -> str:
    """Motor determinístico de contingência: mesmo ciclo ReAct e mesmas ferramentas, sem depender de LLM."""
    msg = mensagem.upper()
    alvos = identificar_tickers(mensagem) or list(ALIASES_TICKERS.keys())
    incluir_backtest = any(p in msg for p in ["BACKTEST", "DESEMPENHO", "HISTÓRIC", "HISTORIC", "ACURÁCIA", "ACURACIA"])

    relatorios, registros = [], []
    for i, t in enumerate(alvos):
        texto, registro = analisar_ticker_offline(t, incluir_backtest, incluir_grafico=(i == 0))
        relatorios.append(texto)
        if registro:
            registros.append(registro)

    if len(registros) > 1:
        comparativo = ["#### 🔀 Comparativo", "| Ticker | Fechamento | RSI | MACD | Sentimento | Recomendação |", "|:-:|:-:|:-:|:-:|:-:|:-:|"]
        comparativo += [f"| {r['ticker']} | R$ {r['close']:.2f} | {r['rsi']:.1f} | {r['macd_signal']} | {r['news_sentiment']:+.2f} | **{r['recommendation']}** |"
                        for r in registros]
        relatorios.append("\n".join(comparativo))

    cabecalho = "> ℹ️ *Resposta gerada pelo Motor Analítico Determinístico (modo resiliente, sem LLM).*"
    return cabecalho + "\n\n" + "\n\n---\n\n".join(relatorios)


async def perguntar_agente(mensagem: str, session_id: str = "sessao_quantum_default") -> str:
    """Envia uma mensagem ao agente ReAct, registra as Actions executadas e retorna a resposta explicável."""
    if ADK_DISPONIVEL:
        try:
            await garantir_sessao(session_id)
            conteudo = types.Content(role="user", parts=[types.Part(text=mensagem)])

            rastro, resposta_final = [], ""
            async for evento in runner.run_async(user_id=USER_ID, session_id=session_id, new_message=conteudo):
                for chamada in (evento.get_function_calls() or []):
                    args = json.dumps(dict(chamada.args or {}), ensure_ascii=False)
                    rastro.append(f"{len(rastro) + 1}. 🛠️ `{chamada.name}({args})`")
                if evento.is_final_response() and evento.content and evento.content.parts:
                    resposta_final += "".join(p.text for p in evento.content.parts if getattr(p, "text", None))

            if resposta_final:
                if rastro:
                    resposta_final += "\n\n---\n#### 🔍 Ferramentas acionadas pelo agente (ReAct trace)\n" + "\n".join(rastro)
                return resposta_final
            print("ℹ️ O agente não retornou texto; acionando o motor analítico.")
        except Exception as e:
            print(f"Executando motor analítico por contingência: {type(e).__name__}: {e}")

    return motor_analitico_offline(mensagem)


## 8. Execução e Emissão de Recomendações para as 4 Ações

Nesta seção, o AI Agent avalia e gera os relatórios completos e a estrutura de dados oficial para as 4 ações obrigatórias do desafio:
- `VALE3`
- `PETR4`
- `BBAS3`
- `ITUB4`


In [ ]:
acoes_desafio = ["VALE3", "PETR4", "BBAS3", "ITUB4"]

print("=" * 80)
print("INICIANDO MONITORAMENTO AUTÔNOMO DAS 4 AÇÕES DA CARTEIRA QUANTUMFINANCE")
print("=" * 80)

for acao in acoes_desafio:
    print(f"\n🔎 Processando análise autônoma para {acao}...")
    resposta = await perguntar_agente(
        f"Realize a análise completa de mercado e emita a recomendação para {acao}.",
        session_id=f"monitoramento_{acao}"
    )
    display(Markdown(resposta))
    print("-" * 80)

# Registro auditável de todas as recomendações emitidas por generate_recommendation
print("📋 Registro consolidado das recomendações (REGISTRO_RECOMENDACOES):")
display(pd.DataFrame(REGISTRO_RECOMENDACOES).drop_duplicates(subset=["ticker", "date"], keep="last"))


## 9. Módulo de Avaliação & Backtest Financeiro (Diferencial de Destaque)

Para atender aos indicadores de avaliação solicitados no enunciado (slide 9):
1. **Acurácia das Recomendações**: percentual de sinais `COMPRAR`/`VENDER` em que o preço seguiu a direção indicada nos 5 pregões seguintes.
2. **Desempenho Financeiro (Backtest)**: retorno acumulado seguindo as recomendações do agente versus a estratégia **Buy-and-Hold** no mesmo período.
3. **Qualidade do Raciocínio**: demonstrada na seção 8 (Thought → Action → Observation + contribuição de cada fator).


In [ ]:
resultados_backtest = []

for acao in acoes_desafio:
    res = run_backtest_strategy(acao, dias_teste=120, horizonte_dias=5)
    if res.get("status") == "sucesso":
        acc = res["acuracia_tendencia_pct"]
        resultados_backtest.append({
            "Ação": res["ticker"],
            "Período": res["periodo"],
            "Retorno Agente (%)": f"{res['retorno_agente_pct']:+.2f}%",
            "Buy & Hold (%)": f"{res['retorno_buy_and_hold_pct']:+.2f}%",
            "Alpha (p.p.)": f"{res['alpha_gerado_pct']:+.2f}",
            "Acurácia de Tendência (5d)": f"{acc:.1f}%" if acc is not None else "n/d",
            "Sinais Avaliados": res["sinais_avaliados"],
            "COMPRAR / VENDER / AGUARDAR": "{COMPRAR} / {VENDER} / {AGUARDAR}".format(**res["distribuicao_sinais"]),
            "Operações": res["total_operacoes"]
        })
    else:
        print(f"⚠️ {acao}: {res.get('mensagem')}")

df_backtest = pd.DataFrame(resultados_backtest)
print("📊 RESUMO CONSOLIDADO DO BACKTEST (ÚLTIMOS 120 PREGÕES, COMPONENTE TÉCNICO):")
display(df_backtest)


## 10. Interface Conversacional Interativa (Gradio)

Implementação da interface conversacional solicitada no enunciado, seguindo o padrão ensinado nas aulas práticas com Gradio (`import gradio as gr`). O painel à direita exibe automaticamente o gráfico gerado pela ferramenta `plot_technical_chart`. Exemplos de perguntas:
- *"Por que você recomendou comprar PETR4 hoje?"*
- *"Mostre o gráfico técnico de VALE3."*
- *"Compare os indicadores de BBAS3 e ITUB4."*
- *"Qual foi a acurácia do backtest de ITUB4?"*


In [ ]:
async def responder_gradio(mensagem: str, historico: list):
    """Conector entre a interface Gradio e o AI Agent ReAct (retorna histórico e último gráfico)."""
    historico = historico or []
    if not mensagem or not mensagem.strip():
        return "", historico, ULTIMO_GRAFICO
    resposta = await perguntar_agente(mensagem, session_id="sessao_gradio")
    historico = historico + [
        {"role": "user", "content": mensagem},
        {"role": "assistant", "content": resposta},
    ]
    return "", historico, ULTIMO_GRAFICO


def criar_chatbot():
    """Cria o componente de chat no formato 'messages' (compatível com Gradio 4.x, 5.x e 6.x)."""
    try:
        return gr.Chatbot(type="messages", height=620, label="QuantumAdvisor")
    except TypeError:
        return gr.Chatbot(height=620, label="QuantumAdvisor")


with gr.Blocks(title="QuantumFinance — AI Investment Assistant") as demo:
    gr.Markdown("""# 🏛️ QuantumFinance — AI Investment Assistant
    ### Assistente Conversacional Autônomo e Explicável para Ações da B3
    *Desenvolvido para o MBA em Data Engineering — FIAP*""")

    with gr.Row():
        with gr.Column(scale=5):
            chatbot = criar_chatbot()
            entrada = gr.Textbox(placeholder="Pergunte sobre VALE3, PETR4, BBAS3 ou ITUB4...", show_label=False)
            with gr.Row():
                enviar = gr.Button("Enviar", variant="primary")
                limpar = gr.Button("Limpar conversa")
        with gr.Column(scale=6):
            grafico = gr.Plot(label="Gráfico técnico (plot_technical_chart)")

    gr.Examples(
        examples=[
            "Qual é a sua recomendação para PETR4 hoje e por quê?",
            "Mostre o gráfico técnico e avalie os indicadores de VALE3.",
            "Compare o momento do Banco do Brasil (BBAS3) e do Itaú (ITUB4).",
            "Qual o sentimento das notícias recentes e o backtest de PETR4?",
        ],
        inputs=entrada,
    )

    enviar.click(responder_gradio, inputs=[entrada, chatbot], outputs=[entrada, chatbot, grafico])
    entrada.submit(responder_gradio, inputs=[entrada, chatbot], outputs=[entrada, chatbot, grafico])
    limpar.click(lambda: ("", [], None), outputs=[entrada, chatbot, grafico])

# Para iniciar a interface no Colab ou localmente:
# demo.launch(share=False)
print("Interface Gradio pronta para execução! Descomente 'demo.launch()' para interagir visualmente.")


## 11. Conclusão & Considerações Finais

### Objetivos Alcançados:
1. **Autonomia com Raciocínio (ReAct)**: O assistente não é um gerador de texto estático, mas sim um agente que decide quais das 6 ferramentas acionar (Thought → Action → Observation) antes de emitir um parecer, com o rastro das ações registrado em cada resposta.
2. **Multifonte Real**: Integração com cotações (`yfinance`) e notícias reais via RSS (`feedparser`), com **sentimento classificado por LLM**, score de impacto por ação e **resumo automático dos eventos**.
3. **Análise Técnica Completa**: RSI, MACD, **SMA e EMA**, Bandas de Bollinger e **Volume** (volume relativo e OBV), conforme o slide 5 do enunciado.
4. **Explicabilidade e Governança**: A decisão é tomada pela ferramenta `generate_recommendation`, com a contribuição de cada fator exposta ao usuário e todas as recomendações registradas para auditoria.
5. **Validação Quantitativa**: Acurácia de tendência e *Alpha* via *Backtest* contra *Buy-and-Hold*, usando o mesmo motor de decisão das recomendações ao vivo.
6. **Interface Conversacional e Visual**: Chat em Gradio com painel de gráfico alimentado pela ferramenta `plot_technical_chart`.

---
*FIAP — MBA em Data Engineering | Disciplina: Agents and Agentic AI*
